In [1]:
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import pyplot
import os
import matplotlib.ticker as ticker
import numpy as np
import xarray as xr
import sys
from utils.utilities import find_best_grid_point, get_station_coords,form_xdate, get_anomalies, adjust_lightness
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import datetime as dt

from plotting import tol_colors # color schemes from https://personal.sron.nl/~pault/
from utils import process_data

#activate interactive figures
%matplotlib widget
#activate autoreload
%load_ext autoreload

## Add parent directory to syspath
parent_dir = os.path.abspath(os.path.join(os.path.dirname('.'), '..'))
if not parent_dir in sys.path:
    sys.path.append(parent_dir)

#save figures in...
dir_save = './output/o3sondes/'

In [3]:
# read in the o3sonde data that Jörg selected for a MKN-pressure level
o3sonde = pl.read_parquet('../data/level2/ecc/nrb_ozone_at_620-624_mbar.parquet')


In [ ]:
## Read all MKN data
%autoreload 2
from input.read_wdc_data import AvailableData, create_data_reader

# File path
data_path = "../data/"

# if New data is added to ./data folder, adapt the dictionary in AvailableData
all_data = list(AvailableData)
print(all_data)

#####---------- TO ADAPT ---------------#####
selected_data = ['CO2', 'CO2_flask', 
                 'CO', 'CO_flask', 
                 'CH4', 'CH4_flask', 
                 'O3'
                 ] # define data to read in. If empty, all data is used 
## 
processing_kwargs = { 
    'FLASK_FLAG_CORR' : True # exclude flagged flask-data
}
#####-----------------------------------#####

datasets = [] # initialize list of all datasets 
# read in data
for sel in (selected_data if selected_data else all_data):
    #define where the data has to be read from
    data_reader =  create_data_reader(data_path=data_path,dataset=sel,**processing_kwargs) #creates an instance of the desired data_reader class
    print(f"Data from {data_reader.__class__.__name__} for {sel}:")

    # call the data-reading function on that instance: 
    data = data_reader.read_data() 
    # call the data-processing
    data = data_reader.process_data(data)

    # prepare merged dataset
    data = data.drop(columns='endtime') # problem when merging datasets (because of NaT?), so better remove endtime
    ds = data.to_xarray()
    ds = ds.assign_coords(dataset=sel)
    ds['species'] = data_reader.species
    ds['unit']  = np.unique(ds.unit.dropna(dim='time'))[0]
    datasets.append(ds)

# save all in one xarray dataset
ds_all = xr.concat(datasets,dim="dataset")


In [ ]:
plt.figure()
ds_all.sel(dataset='O3')['value'].plot(label='MKN')
plt.plot(o3sonde['dtm'],o3sonde['O3_ppmv']*1000,label='o3sonde',marker='.') #in ppb
plt.legend()
plt.show()

In [6]:
df_o3sonde = o3sonde.to_pandas() # note that the dataframe is a polars dataframe (check: print(type(o3sonde['dtm']))! need to convert to pandas for some operations
# Create a pandas DataFrame with 'dtm' as the index
#time_df = pd.DataFrame({'dtm': time})
df_o3sonde.set_index('dtm', inplace=True)
ds_o3sonde = df_o3sonde.to_xarray()

In [ ]:
# Resample the data to daily frequency, but skip days with missing data
ds_o3sonde_daily = ds_o3sonde.resample(dtm='1D').mean()

plt.figure()
#only days with data:
ds_o3sonde_daily.where(ds_o3sonde_daily['O3_ppmv'] > 0, drop=True)['O3_ppmv'].plot(marker='.')
plt.show()

In [ ]:
# Sounding frequency:
nbr_sounding_days = len(ds_o3sonde_daily.where(ds_o3sonde_daily['O3_ppmv'] > 0, drop=True).dtm)
#count the number of weeks and days in the given time period
weeks = len(ds_o3sonde_daily.dtm)/7
print(f"We have in total {nbr_sounding_days} soundings.")
print(f"We have on average {nbr_sounding_days/weeks:.2f} soundings per week.")
print(f"We have on average a sounding every {len(ds_o3sonde_daily.dtm)/nbr_sounding_days:.2f} days.")


In [ ]:
ds_o3sonde_daily.dtm

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ds_all.sel(dataset='O3')['value'].resample(time='1D').mean().plot(label='MKN',marker='',ls='-')
(ds_o3sonde_daily.where(ds_o3sonde_daily['O3_ppmv'] > 0, drop=True)['O3_ppmv']*1000).plot(label='o3sonde',marker='',ls='-') #in ppb
plt.legend()
plt.show()

In [ ]:
## plot curve fit

from utils.ccg_filter import ccg_filter as ccgfilt
from utils.ccg_filter import ccg_dates
from utils import run_curve_fit

## Helper functions for the figure

import matplotlib.colors as mc
from labellines import labelLine, labelLines
import colorsys 

import matplotlib.colors as mc
from labellines import labelLine, labelLines
import colorsys 

# adjust the lightness of a color
def adjust_lightness(color, amount=0.5):
    try:
        c = mc.cnames[color]
    except:
        c = color
    c = colorsys.rgb_to_hls(*mc.to_rgb(c))
    return colorsys.hls_to_rgb(c[0], max(0, min(1, amount * c[1])), c[2])

# function to plot each subplot
# time series plot
def plot_data(ds_temp, ds_temp_fit, label,ax=None, detrend=False, plot_measurements=True, plot_fit=True, **kwargs):
    ''' 
    Plot the data and the fit
    detrend: if True, plot the detrended data
    '''
    if ax is None:
        ax = plt.gca()

    ## get initial figure properties
    initial_color = kwargs['color']
    initial_ls = kwargs['ls']
    initial_zorder = kwargs['zorder']
    
    if plot_measurements:
        kwargs['ls'] = '' #no line for measurements
        pl = ds_temp.plot(
            ax=ax,
            alpha=marker_transp,
            label='',#label[0:4],
            #markeredgewidth=0,
            rasterized=True,
            **kwargs
        )
    else:
        pl = []
    #plot fit
    if plot_fit:
        kwargs['color'] = adjust_lightness(initial_color,amount=1.5) # adapt hue of initial color (lighter)
        kwargs['marker'] = '' #no marker for fit
        kwargs['ls'] = initial_ls
        kwargs['zorder'] = initial_zorder + 1 # plot fit on top of measurements
        pl_fit = ds_temp_fit["smoothed_vals"].plot(ax=ax, label=label, **kwargs)
        if detrend:
            #plot detrended fit
            kwargs['color'] = adjust_lightness(initial_color,amount=0.8)  # adapt hue of initial color (darker)
            kwargs['ls'] = ':' #dotted line for detrended
            ax.plot(
                ds_temp_fit.time,
                ds_temp_fit["seasonal_detrend"] + ds_temp_fit["smoothed_vals"].mean(),
                label="detrended fit",
                **kwargs,
            )  ## Add mean value to detrended to obtain same magnitude
    else:
        pl_fit = []
    return pl, pl_fit

# seasonality plot
def plot_cycle(ds_temp, ds_temp_fit, s, freq="month", ax=None, with_trend = False, plot_smoothed=True, label='', **kwargs):
    '''
    Plot seasonal cycle of the data
    with_trend:  if True, plot the seasonal cycle with trend (non-detrended)
    plot_smootehd:  Plot the smoothed or the original data
    '''
    
    ## get a different hue of the color used
    # Decrease the hue value
    initial_color = kwargs['color']
    
    if ax is None:
        ax = plt.gca()
    ref = (
        ds_temp.mean()
    )  # use mean value from measurements to obtain positive/neg. seasonality (ds_temp-ref) or absolute values (ds_temp_fit +ref)

    if freq=='hour':
        kwargs['color'] = adjust_lightness(initial_color,amount=1.2)  # adapt hue of initial color (lighter)
        
        xvals = (ds_temp-ref).groupby(f"time.{freq}").mean()
        pl = xvals.plot(
            ax=ax, label=f"{label}",**kwargs
        )
        std = (ds_temp-ref).groupby(f"time.{freq}").std() # standarddeviation of the grouped values of the measurements
        #add stdev
        ax.fill_between(xvals[freq],xvals,xvals+std,alpha=.3,label=f'std. dev.',color=kwargs['color'])
        ax.fill_between(xvals[freq],xvals,xvals-std,alpha=.3,label=f'std. dev.',color=kwargs['color'])
    else:
        # normal seasonal cycle
        if with_trend:
            kwargs['ls'] = ':' #dotted line for not detrended
            kwargs['color'] = adjust_lightness(initial_color,amount=0.8) # adapt hue of initial color (darker)
            #plot also the normal seasonal cycle (with trend) in addition
            (ds_temp-ref).groupby(f"time.{freq}").mean().plot(ax=ax, label=f"{label} not detrended",**kwargs)
        # detrended seasonal cycle
        kwargs['color'] = adjust_lightness(initial_color,amount=1.2)  # adapt hue of initial color (lighter)

        if plot_smoothed:
            xvals = ds_temp_fit["seasonal_detrend"].groupby(f"time.{freq}").mean()
            std = ds_temp_fit["seasonal_detrend"].groupby(f"time.{freq}").std() # standarddeviation of the grouped values of the measurements
        else:
            xvals = (ds_temp-ref).groupby(f"time.{freq}").mean()
            std = (ds_temp-ref).groupby(f"time.{freq}").std() 
            

        pl = xvals.plot(
            ax=ax, label=f"{label}", **kwargs
        ) #label=f"{label} detrended"

        #add stdev
        ax.fill_between(xvals[freq],xvals,xvals+std,alpha=.3,label=f'std. dev.',color=kwargs['color'])
        ax.fill_between(xvals[freq],xvals,xvals-std,alpha=.3,label=f'std. dev.',color=kwargs['color'])
    return pl

In [ ]:
# Define fit paramaters for the curve fitting
# Default values
fit_params_defaults = {'shortterm': 80, #Short term cutoff value in days for smoothing of data
                'longterm': 667, # smoothing in days. Default: 667
                'numpolyterms': 3, # use only 2 for less than 3 years of data, otherwise use 3 (=quadratic fit)
                'sampleinterval': 1 / 24,  # 1h
                'numharmonics': 4}


fit_properties = {}
for dataset_name in ['O3','o3sonde']:
    fit_properties[dataset_name] = fit_params_defaults.copy()

In [ ]:

o3_daily  = ds_all.sel(dataset='O3')['value'].resample(time='1D').mean()
o3sonde_daily = (ds_o3sonde_daily.where(ds_o3sonde_daily['O3_ppmv'] > 0, drop=True)['O3_ppmv']*1000) #in ppb
tsel1 = '1997-01-01'
tsel2 = '2023-12-31'

fig, ax = plt.subplots(1, 1, figsize=(10, 10/16/9), layout="constrained")

# this would be local activity only
# gfas_mean["frpfire"].plot(label="Total mean fire activity", color="C1")
# gfas_mean_local["frpfire"].plot(label=f"Local fire activity (+/-{dlat}°lat/{dlon}°lon)", color="C3", ls=":")

##----------------- Plot O3 data -----------------##
o3_daily.plot(
    color=adjust_lightness('b', amount=1.5),
    label="",
    marker="o",
    ls="",
    alpha=0.5,
    markeredgewidth=0,
)
o3sonde_daily.plot(
    color=adjust_lightness('r', amount=1.5),
    label="",
    marker="o",
    ls="",
    alpha=0.5,
    markeredgewidth=0,
)
## Plot curve fit
for ds, ds_str in zip([o3_daily, o3sonde_daily], ['O3','o3sonde']):
    filt, df_interp, ds_interp = run_curve_fit.run_ccgfilter(
        ds=ds, dataset_str=ds_str, t1=tsel1, t2=tsel2, **fit_properties[ds_str]
    )
    # select in the plot() functin fire_color2 if ds_str == "fire_south" else fire_color:

    ds_interp["smoothed_vals"].plot(ax=ax, label=ds_str.replace('_',' '), lw=lw_thick)
plt.title("")
# Axes properties
ax.set_ylabel("Ozone (ppb)")
ax.legend(loc="upper right")
ax.set_title("Daily ozoe values at Mt. Kenya", loc="left")

# write the ylabels in scientific notation
# ax.yaxis.set_major_formatter(ticker.ScalarFormatter(useMathText=True))
plt.ticklabel_format(style="sci", axis="y", scilimits=(0, 0))
